In [1]:
# ==============================================================================
# INITIALISATION ET IMPORTATION DES LIBRAIRIES
# ==============================================================================
import os
import re
import string
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk

# --- Outils de Traitement du Langage (NLP) et Similarité ---
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from wordcloud import WordCloud

# --- Machine Learning ---
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# --- Explainable AI ---
import shap

# --- Configuration visuelle et environnement ---
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
sns.set_theme(style="whitegrid")

# Téléchargement des dictionnaires NLTK
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('vader_lexicon', quiet=True)

print("✅ Cellule 1 terminée : Tous les outils sont chargés.")

✅ Cellule 1 terminée : Tous les outils sont chargés.


In [3]:
def auditer_et_corriger_dataset(dataframe):
    print("--- DÉBUT DE L'AUDIT DE QUALITÉ ---")
    df_propre = dataframe.copy()
    
    taille_avant = len(df_propre)
    df_propre = df_propre.dropna(subset=['text_', 'label'])
    if taille_avant - len(df_propre) > 0:
        print(f"🔧 CORRECTION : {taille_avant - len(df_propre)} lignes nulles supprimées.")

    taille_avant = len(df_propre)
    df_propre = df_propre[df_propre['label'].isin(['CG', 'OR'])]
    if taille_avant - len(df_propre) > 0:
        print(f"🔧 CORRECTION : {taille_avant - len(df_propre)} labels aberrants exclus.")

    taille_avant = len(df_propre)
    df_propre['text_'] = df_propre['text_'].replace(r'^\s*$', np.nan, regex=True)
    df_propre = df_propre.dropna(subset=['text_'])
    
    print(f"🟢 DATASET VALIDÉ : Taille finale = {len(df_propre)} lignes.")
    return df_propre

df = pd.read_csv('fake reviews dataset.csv')
df = auditer_et_corriger_dataset(df)
df = df.drop_duplicates().reset_index(drop=True)

--- DÉBUT DE L'AUDIT DE QUALITÉ ---
🟢 DATASET VALIDÉ : Taille finale = 40432 lignes.


In [1]:
print("\n--- GÉNÉRATION DU TABLEAU DE BORD (EDA) ---")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
sns.countplot(data=df, x='label', ax=axes[0], palette=['#3498db', '#e74c3c'], hue='label', legend=False)
axes[0].set_title('Répartition des Avis (CG = Fraude, OR = Original)', fontweight='bold')
 
sns.countplot(data=df, x='rating', hue='label', ax=axes[1], palette=['#3498db', '#e74c3c'])
axes[1].set_title('Distribution des Notes', fontweight='bold')
 
df['longueur_mots'] = df['text_'].apply(lambda x: len(str(x).split()))
sns.histplot(data=df, x='longueur_mots', hue='label', bins=50, kde=True, ax=axes[2], palette=['#3498db', '#e74c3c'])
axes[2].set_title('Longueur des Avis (en mots)', fontweight='bold')
axes[2].set_xlim(0, 200)
 
plt.tight_layout()
plt.show()
 
df = df.drop(columns=['longueur_mots'])


--- GÉNÉRATION DU TABLEAU DE BORD (EDA) ---


NameError: name 'plt' is not defined

In [2]:
print("--- GÉNÉRATION DES NUAGES DE MOTS ---")
 
textes_fraude = " ".join(text for text in df[df['label'] == 'CG']['text_'].dropna())
textes_originaux = " ".join(text for text in df[df['label'] == 'OR']['text_'].dropna())
 
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
 
wordcloud_cg = WordCloud(width=800, height=400, background_color='white', colormap='Reds', max_words=100).generate(textes_fraude)
axes[0].imshow(wordcloud_cg, interpolation='bilinear')
axes[0].set_title('Mots les plus fréquents - FRAUDES (CG)', fontsize=14, fontweight='bold')
axes[0].axis('off')
 
wordcloud_or = WordCloud(width=800, height=400, background_color='white', colormap='Blues', max_words=100).generate(textes_originaux)
axes[1].imshow(wordcloud_or, interpolation='bilinear')
axes[1].set_title('Mots les plus fréquents - ORIGINAUX (OR)', fontsize=14, fontweight='bold')
axes[1].axis('off')
 
plt.tight_layout()
plt.show()

--- GÉNÉRATION DES NUAGES DE MOTS ---


NameError: name 'df' is not defined

In [ ]:
lemmatizer = WordNetLemmatizer()

sia = SentimentIntensityAnalyzer()
 
def clean_text_avance(text):

    text = str(text).lower()

    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

    text = re.sub(r'\d+', '', text)

    mots = text.split()

    mots_lemmatises = [lemmatizer.lemmatize(mot) for mot in mots]

    return ' '.join(mots_lemmatises)
 
print("1. Nettoyage et Lemmatisation en cours...")

df['text_cleaned'] = df['text_'].apply(clean_text_avance)

df['target'] = df['label'].apply(lambda x: 1 if x == 'CG' else 0)
 
print("2. Création des Méta-Features (statistiques sur la structure)...")

df['word_count'] = df['text_'].apply(lambda x: len(str(x).split()))

df['char_count'] = df['text_'].apply(lambda x: len(str(x)))

df['caps_count'] = df['text_'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

df['punct_count'] = df['text_'].apply(lambda x: sum(1 for c in str(x) if c in string.punctuation))
 
print("3. Création des scores de sentiments (Analyse VADER)...")

def get_sentiment_score(text):

    return sia.polarity_scores(str(text))['compound']
 
df['sentiment_score'] = df['text_'].apply(get_sentiment_score)

df['is_extreme_boolean'] = df['sentiment_score'].abs() > 0.95
 
display(df[['label', 'word_count', 'caps_count', 'sentiment_score', 'is_extreme_boolean']].head())
 

In [ ]:
print("--- DÉMONSTRATION SIMILARITÉ SENTENCE-BERT ---")
 
df_sim = df.sample(n=1000, random_state=42).reset_index(drop=True)
 
model_sim = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_sim.encode(df_sim['text_cleaned'].tolist(), show_progress_bar=False)
 
matrice_similarite = cosine_similarity(embeddings)
 
index_cible = 0
scores = matrice_similarite[index_cible]
index_similaires = np.argsort(scores)[::-1][1:3]
 
print(f"\n🔍 AVIS ORIGINAL (Label: {df_sim['label'].iloc[index_cible]}) : '{df_sim['text_'].iloc[index_cible]}'")
print("🔥 AVIS AYANT LE MÊME SENS DÉTECTÉS PAR BERT :")
for idx in index_similaires:
    print(f" -> Score de similarité: {scores[idx]:.3f} | Label: {df_sim['label'].iloc[idx]}")
    print(f"    Texte: '{df_sim['text_'].iloc[idx]}'")

In [ ]:
class BertTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)
        return self.model.encode(X.tolist(), show_progress_bar=False)
 
df_bench = df.sample(n=2500, random_state=42).reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(df_bench['text_cleaned'], df_bench['target'], test_size=0.2, random_state=42)
 
pipelines = {
    "BoW + Naive Bayes": Pipeline([('vectorizer', CountVectorizer(max_features=5000)), ('classifier', MultinomialNB())]),
    "TF-IDF + LogReg": Pipeline([('vectorizer', TfidfVectorizer(max_features=5000)), ('classifier', LogisticRegression(max_iter=1000))]),
    "TF-IDF + LinearSVC": Pipeline([('vectorizer', TfidfVectorizer(max_features=5000)), ('classifier', LinearSVC(random_state=42, dual=False))]),
    "BERT + LogReg": Pipeline([('vectorizer', BertTransformer()), ('classifier', LogisticRegression(max_iter=1000))]),
    "BERT + Random Forest": Pipeline([('vectorizer', BertTransformer()), ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])
}
 
results = {}
print("Entraînement des pipelines en cours...")
for name, pipeline in pipelines.items():
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc, f1 = accuracy_score(y_test, preds), f1_score(y_test, preds)
    results[name] = {"Accuracy": acc, "F1-Score": f1, "Pipeline": pipeline}
    print(f"✅ {name: <25} | F1: {f1:.4f} | Acc: {acc:.4f} | Temps: {time.time() - start_time:.2f}s")
 
res_df = pd.DataFrame(results).T.sort_values(by='F1-Score', ascending=False)
best_pipe_name = res_df.index[0]
best_pipeline = results[best_pipe_name]["Pipeline"]
 
plt.figure(figsize=(10, 5))
colors = ['#9b59b6' if 'BERT' in x else '#3498db' for x in res_df.index]
res_df['F1-Score'].plot(kind='bar', color=colors, edgecolor='black')
plt.title('Comparaison F1-Score (Pipelines)', fontweight='bold')
plt.ylim(0.60, 1.0)
plt.xticks(rotation=30, ha='right')
plt.show()

In [ ]:
# ==============================================================================
# PARTIE 6 : OPTIMISATION ULTIME (HYBRIDE DYNAMIQUE + GRID SEARCH)
# ==============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import joblib
 
print(f"\n--- 🌟 OPTIMISATION ULTIME SUR LE GAGNANT : {best_pipe_name.upper()} ---")
 
# 1. On prépare les données complètes (Texte + Variables numériques de la Partie 4)
colonnes_numeriques = ['word_count', 'caps_count', 'sentiment_score', 'is_extreme_boolean']
X_hyb = df[['text_cleaned'] + colonnes_numeriques]
y_hyb = df['target']
 
# Séparation des données (80% / 20%)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_hyb, y_hyb, test_size=0.2, random_state=42)
 
# 2. EXTRACTION DYNAMIQUE : On récupère les outils précis du pipeline gagnant
vect_gagnant = best_pipeline.named_steps['vectorizer']
clf_gagnant = best_pipeline.named_steps['classifier']
 
# 3. Le Chef d'Orchestre (ColumnTransformer)
preprocesseur = ColumnTransformer(
    transformers=[
        # On applique le vectoriseur gagnant (TF-IDF ou BERT) uniquement sur le texte
        ('texte', vect_gagnant, 'text_cleaned'),
        # On normalise les variables numériques
        ('nombres', StandardScaler(), colonnes_numeriques)
    ])
 
# 4. Création du Pipeline Hybride
pipeline_hybride = Pipeline([
    ('preprocesseur', preprocesseur),
    ('classifier', clf_gagnant)
])
 
# 5. La Grille des paramètres qui s'adapte automatiquement
param_grid_hybride = {}
 
# Si TF-IDF a gagné, on optimise le vocabulaire (Si BERT a gagné, on n'y touche pas car c'est trop lourd)
if 'Tfidf' in str(type(vect_gagnant)):
    param_grid_hybride['preprocesseur__texte__max_features'] = [10000, 20000]
    param_grid_hybride['preprocesseur__texte__ngram_range'] = [(1, 1), (1, 2)]
 
# Optimisation du classifieur selon celui qui a gagné
if 'LogisticRegression' in str(type(clf_gagnant)):
    param_grid_hybride['classifier__C'] = [0.1, 1.0, 10.0]
    param_grid_hybride['classifier__max_iter'] = [1000]
elif 'LinearSVC' in str(type(clf_gagnant)):
    param_grid_hybride['classifier__C'] = [0.1, 0.5, 1.0]
elif 'RandomForest' in str(type(clf_gagnant)):
    param_grid_hybride['classifier__n_estimators'] = [100, 200]
 
print("L'ordinateur cherche la meilleure combinaison Texte + Variables Numériques...")
 
# 6. Lancement du Grid Search
grid_hybride = GridSearchCV(pipeline_hybride, param_grid_hybride, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_hybride.fit(X_train_h, y_train_h)
 
# 7. Affichage des résultats
modele_ultime = grid_hybride.best_estimator_
print(f"\n🏆 Meilleurs paramètres trouvés : {grid_hybride.best_params_}")
print(f"Meilleur F1-Score en validation : {grid_hybride.best_score_:.4f}")
 
predictions_h = modele_ultime.predict(X_test_h)
 
plt.figure(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_test_h, predictions_h), annot=True, fmt='d', cmap='Greens', 
            xticklabels=['Original', 'Fraude'], yticklabels=['Original', 'Fraude'], linewidths=1, linecolor='black')
plt.title(f'Matrice de Confusion : HYBRIDE ULTIME ({best_pipe_name})', fontsize=14, fontweight='bold')
plt.ylabel('Vrai Label', fontsize=12)
plt.xlabel('Prédiction', fontsize=12)
plt.show()
 
# 8. Exportation
joblib.dump(modele_ultime, 'pipeline_ultime_hybride.pkl')
print("✅ Modèle hybride parfait exporté sous 'pipeline_ultime_hybride.pkl' !")